# 07_01 The corpus: what is in data/homer?

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> [Odyssey text] → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **Odyssey text**
>
> Project files this connects to:
>
> - `data/homer/README.md`
> - `data/homer/iliad/`
> - `data/homer/odyssey/`
>
> **This notebook:** Before training on a second poem, we should know what is in `data/homer/`: how big each text is, how many characters it uses, and where the two Greek poems differ.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How big each poem is, in books, lines and characters
- How many different characters each text uses, and why Greek needs so many more
- Which characters only one Greek poem has
- How the two poems begin, word by word

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **corpus** | a collection of texts that we study or train on | the 4 texts in `data/homer/` |
| **edition** | one scholar's prepared version of an ancient text: the old manuscripts disagree in places, so editors choose | Murray's Odyssey, Monro and Allen's Iliad |
| **unicodedata.name** | a built-in lookup that gives a character its official name | `unicodedata.name("ἄ")` is `GREEK SMALL LETTER ALPHA WITH PSILI AND OXIA` |

## 3. The idea

### ❓ What's in the `data/homer/` folder?

Four texts: two poems, each in Ancient Greek and in English, 24 books each, one file per book ([data/homer/README.md](../../data/homer/README.md) has the sources and licences).

```
data/homer/
  odyssey/greek/    book_01.txt … book_24.txt     one verse per line
  odyssey/english/  book_01.txt … book_24.txt     one paragraph per line
  iliad/greek/      book_01.txt … book_24.txt     one verse per line
  iliad/english/    book_01.txt … book_24.txt     one paragraph per line
```

The Greek is the original verse. The English is Samuel Butler's prose translation, so its "lines" are whole paragraphs. Training reads this folder too, but by default only `odyssey/greek/book_01.txt`. This chapter is the first to use the other books and texts.

👉 **Predict first:** which text is the biggest, in characters? Does the English or the Greek have more?

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ How big is each text?

We read all 24 files of each folder and count the books, the lines (a line ends with `\n`) and the characters (what `len()` counts, including the `\n`).

In [ ]:
# data/homer: load_book_files reads all the books of a folder
from homer_ai.lessons import HOMER, load_book_files

In [ ]:
corpus = {}  # (poem, language) -> the list of 24 book texts
for poem in ("odyssey", "iliad"):
    for language in ("greek", "english"):
        corpus[(poem, language)] = load_book_files(HOMER / poem / language)

print(f"{'text':16} {'books':>5} {'lines':>7} {'characters':>11} {'per line':>9}")
for (poem, language), books in corpus.items():
    lines = sum(book.count("\n") for book in books)
    characters = sum(len(book) for book in books)
    print(
        f"{poem + ' ' + language:16} {len(books):>5} {lines:>7,} "
        f"{characters:>11,} {characters / lines:>9.1f}"
    )

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, GOOD, INK, studio_axes, studio_figure

title = "How much text is there? (thousands of characters)"
fig = studio_figure("the corpus", title, (10, 5.5))
ax = studio_axes(fig, (0.1, 0.12, 0.8, 0.64))
width = 0.35
for n, (poem, colour) in enumerate([("odyssey", ACCENT), ("iliad", GOOD)]):
    sizes = [
        sum(len(book) for book in corpus[(poem, language)]) / 1000
        for language in ("greek", "english")
    ]
    positions = [i + (n - 0.5) * width for i in range(2)]
    ax.bar(positions, sizes, width, color=colour, label=poem.capitalize())
    for x, size in zip(positions, sizes, strict=True):
        ax.text(x, size + 10, f"{size:,.0f}", ha="center", color=INK,
                family="monospace", fontsize=11, fontweight="bold")
ax.set_xticks([0, 1], ["Greek", "English (Butler)"])
ax.tick_params(axis="x", labelsize=13, labelcolor=INK)
ax.set_ylabel("thousands of characters")
ax.set_ylim(0, 950)
ax.grid(axis="x", visible=False)
ax.set_axisbelow(True)
ax.legend(frameon=False, labelcolor=INK)
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

In the table and the picture, the Iliad is bigger than the Odyssey (15,687 Greek verses against 12,107). The English has **more** characters than the Greek (606,758 against 538,303 for the Odyssey) but far fewer lines. That's about 590 characters per line, against about 44 for Greek verse. A Greek line is one verse, and an English line is a whole paragraph. [07_05](07_05_english_and_greek.ipynb) comes back to this.

</details>

### ❓ How many different characters does each text use?

This is the size of the character vocabulary ([02_01](../02_text_to_numbers/02_01_tokens_and_vocabulary.ipynb)), before the two special tokens. We keep each character once, as in 02_01. A line break counts as a character too.

👉 **Predict first:** English has 26 letters, capitals, digits and punctuation. Greek has 24 letters. Which will have more different characters, and by how many times?

In [ ]:
distinct = {key: set("".join(books)) for key, books in corpus.items()}
for (poem, language), characters in distinct.items():
    print(f"{poem + ' ' + language:16} {len(characters):>4} different characters")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

167 (Odyssey) and 169 (Iliad) for Greek, against 67 and 68 for English. Greek needs more than **twice** as many. The reason is the accents. A Greek letter can carry a breathing mark, an accent, or both, and each combination is its own character. Here's `α`:

</details>

In [ ]:
import unicodedata

odyssey_greek = distinct[("odyssey", "greek")]
alphas = sorted(c for c in odyssey_greek if "ALPHA" in unicodedata.name(c, ""))
print(len(alphas), "different alpha characters in the Odyssey:", " ".join(alphas))
print()
print("ἄ is called:", unicodedata.name("ἄ"))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

That's 18 different alphas in the Odyssey alone, each its own character. The name spells out the parts (`PSILI` is the gentle breathing mark, `OXIA` the sharp accent). English has one `a` and one `A`. That's why a Greek model needs a bigger vocabulary, and why the numbers of tokens differ later in [07_05](07_05_english_and_greek.ipynb).

</details>

### ❓ Which characters are in one Greek poem but not the other?

We compare the two character sets and look for what's in one poem and not in the other. We also count how often each one occurs, and print its name.

👉 **Predict first:** the two poems are in the same language, so the sets should be nearly equal. Will `Iliad − Odyssey` be empty?

In [ ]:
odyssey_chars = distinct[("odyssey", "greek")]
iliad_chars = distinct[("iliad", "greek")]
iliad_text = "".join(corpus[("iliad", "greek")])
odyssey_text = "".join(corpus[("odyssey", "greek")])

print("only in the Iliad:")
for c in sorted(iliad_chars - odyssey_chars):
    print(f"  {c}  {iliad_text.count(c):>3} times  {unicodedata.name(c)}")
print("only in the Odyssey:")
for c in sorted(odyssey_chars - iliad_chars):
    print(f"  {c}  {odyssey_text.count(c):>3} times  {unicodedata.name(c)}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Six characters appear only in the Iliad, and four only in the Odyssey. They're all rare (some occur once or twice in a whole poem), but a tokenizer built from one poem has **no number** for them. That's the trap of [07_03](07_03_comparing_fairly.ipynb). And one of the four Odyssey-only characters isn't Greek at all. It's the long dash `—`.

</details>

### ❓ How do the two poems begin?

Each poem starts with a request to the Muse. Here's the first line of each, word by word, from the real files. The English is the start of Butler's paragraph, and the glosses are our own.

```
μῆνιν    ἄειδε   θεὰ      Πηληϊάδεω     Ἀχιλῆος
anger    sing    goddess  of-Peleus'-son  of-Achilles

ἄνδρα    μοι     ἔννεπε,  μοῦσα,   πολύτροπον,
man      me      tell     Muse     of-many-turns
```

👉 **Predict first:** the code prints the same two rows. Which poem starts with a man, and which with an anger?

In [ ]:
GLOSSES = {
    "iliad": ["anger", "sing", "goddess", "of-Peleus'-son", "of-Achilles"],
    "odyssey": ["man", "me", "tell", "Muse", "of-many-turns"],
}
for poem in ("iliad", "odyssey"):
    first_line = corpus[(poem, "greek")][0].splitlines()[0]
    greek_words = first_line.split()[:5]
    print(f"{poem.upper()}  (verse 1 of book 1)")
    print("   ".join(f"{w:<14}" for w in greek_words))
    print("   ".join(f"{g:<14}" for g in GLOSSES[poem]))
    english = corpus[(poem, "english")][0].splitlines()[0]
    print("Butler:", english[:80], "…\n")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The Iliad starts with `μῆνιν` ("anger"), the Odyssey with `ἄνδρα` ("man"). So each poem names its subject in its first word. We use these two words as prompts in [07_04](07_04_combining_both.ipynb), to see whether a model that knows both poems answers differently.

</details>

### ❓ Why do the extra characters matter for a model?

A model has one row of numbers for every token in its vocabulary (the embedding table of [02_05](../02_text_to_numbers/02_05_token_embeddings.ipynb), and the output head). A different vocabulary means a different-shaped model, and the same letter can get a different number. Two models built that way can't be compared directly. [07_03](07_03_comparing_fairly.ipynb) shows it with real numbers.

## 5. Recap

- `data/homer/` has four texts: 24 books each of Greek verse and Butler's English prose, for the Odyssey and the Iliad.
- Greek uses about 167 different characters against English's 67, because every accent combination is its own character.
- Six characters appear only in the Iliad and four only in the Odyssey. It's a small thing, but it matters for fair comparisons.

**Next:** [07_02: An Iliad model](07_02_an_iliad_model.ipynb).